# 1. Project overview

This Google Colab notebook builds a document-grounded AI assistant for **The Happy Prince and Other Tales** by Oscar Wilde. It validates and extracts the supplied PDF, creates semantic chunks by comparing adjacent OpenAI embeddings, stores explicit embeddings and page metadata in one ChromaDB collection, and routes natural-language requests to text, audio, or image output.

Run the notebook from top to bottom. Add secrets through the Colab key icon (or environment variables); no credentials are stored in this notebook.

# 2. Dependency installation

In [ ]:
%pip install -q -U openai chromadb pypdf pydantic elevenlabs numpy

# 3. Imports and configuration

In [ ]:
import base64
import hashlib
import os
import re
import uuid
from pathlib import Path
from typing import Any, Literal, Sequence

import chromadb
import numpy as np
from elevenlabs.client import ElevenLabs
from IPython.display import Audio, Image, display
from openai import OpenAI
from pydantic import BaseModel, ConfigDict
from pypdf import PdfReader

PDF_SIZE_LIMIT_BYTES = 10 * 1024 * 1024
COLLECTION_NAME = "oscar_wilde_semantic_chunks"
CHROMA_DIR = Path("/content/chroma_oscar_wilde")
GENERATED_DIR = Path("/content/generated")

EMBEDDING_MODEL = "text-embedding-3-small"
TEXT_MODEL = "gpt-5.6-luna"
IMAGE_MODEL = "gpt-image-2"
ELEVENLABS_MODEL = "eleven_multilingual_v2"

MIN_CHUNK_CHARS = 500
MAX_CHUNK_CHARS = 1800
OVERLAP_CHARS = 200
BREAKPOINT_PERCENTILE = 85.0
RETRIEVAL_TOP_K = 5

GENERATED_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
print("Configuration loaded. Generated files:", GENERATED_DIR)

# 4. Secret loading

Secrets are read from Google Colab Secrets first, then environment variables. Values are never printed. OpenAI is required for indexing and retrieval; ElevenLabs credentials are loaded only when audio is requested.

In [ ]:
def load_secret(name: str, *, required: bool = False) -> str | None:
    """Load a secret from Colab userdata or an environment variable."""
    value: str | None = None
    try:
        from google.colab import userdata

        value = userdata.get(name)
    except ImportError:
        value = None
    except Exception:
        # Colab raises when a secret is missing or access was not granted.
        value = None

    value = value or os.getenv(name)
    if required and not value:
        raise RuntimeError(
            f"Missing {name}. Add it to Google Colab Secrets or set it as an environment variable."
        )
    return value


OPENAI_API_KEY = load_secret("OPENAI_API_KEY", required=True)
openai_client = OpenAI(api_key=OPENAI_API_KEY)
print("OpenAI credentials loaded successfully (value hidden).")

# 5. PDF validation and extraction

In [ ]:
PDF_CANDIDATES = (
    Path("/sample_date/The_Happy_Prince_and_Other_Tales_by_Oscar_Wilde.pdf"),
    Path("/content/sample_data/The_Happy_Prince_and_Other_Tales_by_Oscar_Wilde.pdf"),
    Path("/content/sample_data/The_Happy_Prince,_and_Other_Tales_by_Oscar_Wilde.pdf"),
    Path("sample_data/The_Happy_Prince_and_Other_Tales_by_Oscar_Wilde.pdf"),
    Path("sample_data/The_Happy_Prince,_and_Other_Tales_by_Oscar_Wilde.pdf"),
)


def resolve_pdf_path(candidates: Sequence[Path]) -> Path:
    """Return the first existing PDF candidate or raise a clear error."""
    for candidate in candidates:
        expanded = candidate.expanduser()
        if expanded.is_file():
            return expanded.resolve()
    tried = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(
        "Oscar Wilde PDF not found. Upload it to one of these locations:\n" + tried
    )


def clean_page_text(text: str) -> str:
    """Normalize extracted text while retaining paragraph boundaries."""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"(?<=\w)-\n(?=\w)", "", text)
    paragraphs = []
    for block in re.split(r"\n\s*\n", text):
        block = re.sub(r"[ \t]+", " ", block)
        block = re.sub(r"\s*\n\s*", " ", block).strip()
        if block:
            paragraphs.append(block)
    return "\n\n".join(paragraphs)


def validate_and_extract_pdf(pdf_path: Path) -> tuple[list[dict[str, Any]], float]:
    """Validate a PDF and return nonempty page text with one-based page numbers."""
    if not pdf_path.is_file():
        raise FileNotFoundError(f"PDF does not exist: {pdf_path}")
    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError(f"Expected a .pdf file, received: {pdf_path.name}")

    size_bytes = pdf_path.stat().st_size
    if size_bytes <= 0:
        raise ValueError("The PDF is empty.")
    if size_bytes >= PDF_SIZE_LIMIT_BYTES:
        raise ValueError(
            f"PDF is {size_bytes / (1024 ** 2):.2f} MB; it must be smaller than 10 MB."
        )

    try:
        reader = PdfReader(str(pdf_path))
    except Exception as exc:
        raise ValueError(f"The PDF could not be opened: {exc}") from exc
    if not reader.pages:
        raise ValueError("The PDF contains no pages.")

    pages: list[dict[str, Any]] = []
    extraction_errors: list[int] = []
    for page_number, page in enumerate(reader.pages, start=1):
        try:
            text = clean_page_text(page.extract_text() or "")
        except Exception:
            text = ""
            extraction_errors.append(page_number)
        if text:
            pages.append({"page_number": page_number, "text": text})

    if not pages or not any(page["text"].strip() for page in pages):
        raise ValueError(
            "The PDF contains no extractable text. It may be scanned and require OCR."
        )

    size_mb = size_bytes / (1024 ** 2)
    print(f"Resolved PDF: {pdf_path}")
    print(f"Validated size: {size_mb:.2f} MB (< 10 MB)")
    print(f"Extracted text from {len(pages)} of {len(reader.pages)} pages.")
    if extraction_errors:
        print(f"Warning: extraction errors on {len(extraction_errors)} page(s).")
    return pages, size_mb


PDF_PATH = resolve_pdf_path(PDF_CANDIDATES)
EXTRACTED_PAGES, PDF_SIZE_MB = validate_and_extract_pdf(PDF_PATH)

# 6. Semantic chunking

Text is first converted into paragraph or sentence units. Adjacent unit embeddings are compared with cosine distance. A boundary is created at a high, document-adaptive distance percentile when the minimum size has been reached, while the maximum size is always enforced.

In [ ]:
SENTENCE_BOUNDARY = re.compile(r"(?<=[.!?])\s+(?=[A-Z\"'“‘])")


def split_long_text(text: str, max_chars: int = 450) -> list[str]:
    """Split a paragraph into sentence/word units no larger than max_chars when possible."""
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    sentences = [part.strip() for part in SENTENCE_BOUNDARY.split(text) if part.strip()]
    units: list[str] = []
    for sentence in sentences:
        if len(sentence) <= max_chars:
            units.append(sentence)
            continue
        words = sentence.split()
        current: list[str] = []
        for word in words:
            candidate = " ".join([*current, word])
            if current and len(candidate) > max_chars:
                units.append(" ".join(current))
                current = [word]
            else:
                current.append(word)
        if current:
            units.append(" ".join(current))
    return units


def create_text_units(pages: Sequence[dict[str, Any]]) -> list[dict[str, Any]]:
    """Create ordered paragraph/sentence units and preserve their page numbers."""
    units: list[dict[str, Any]] = []
    for page in pages:
        paragraphs = re.split(r"\n\s*\n", page["text"])
        for paragraph in paragraphs:
            for text in split_long_text(paragraph):
                units.append(
                    {
                        "unit_index": len(units),
                        "page_number": int(page["page_number"]),
                        "text": text,
                    }
                )
    if not units:
        raise ValueError("No semantic text units could be created from the PDF.")
    return units


def cosine_distance(left: Sequence[float], right: Sequence[float]) -> float:
    """Return cosine distance in the range normally bounded by 0 and 2."""
    left_vector = np.asarray(left, dtype=np.float32)
    right_vector = np.asarray(right, dtype=np.float32)
    denominator = float(np.linalg.norm(left_vector) * np.linalg.norm(right_vector))
    if denominator == 0.0:
        return 1.0
    similarity = float(np.dot(left_vector, right_vector) / denominator)
    return 1.0 - max(-1.0, min(1.0, similarity))


def _chunk_char_count(units: Sequence[dict[str, Any]]) -> int:
    return sum(len(unit["text"]) for unit in units) + max(0, len(units) - 1)


def semantic_chunk_units(
    units: Sequence[dict[str, Any]],
    embeddings: Sequence[Sequence[float]],
    *,
    min_chars: int = MIN_CHUNK_CHARS,
    max_chars: int = MAX_CHUNK_CHARS,
    overlap_chars: int = OVERLAP_CHARS,
    breakpoint_percentile: float = BREAKPOINT_PERCENTILE,
) -> tuple[list[dict[str, Any]], float, list[float]]:
    """Create size-bounded semantic chunks with whole-unit overlap."""
    if len(units) != len(embeddings):
        raise ValueError("Every semantic unit must have exactly one embedding.")
    if not units:
        raise ValueError("Cannot chunk an empty unit list.")

    distances = [
        cosine_distance(embeddings[index], embeddings[index + 1])
        for index in range(len(embeddings) - 1)
    ]
    threshold = float(np.percentile(distances, breakpoint_percentile)) if distances else 1.0

    base_chunks: list[list[dict[str, Any]]] = []
    current: list[dict[str, Any]] = []
    for index, unit in enumerate(units):
        projected_size = _chunk_char_count([*current, unit])
        if current and projected_size > max_chars:
            base_chunks.append(current)
            current = [unit]
        else:
            current.append(unit)

        semantic_break = index < len(distances) and distances[index] >= threshold
        if semantic_break and _chunk_char_count(current) >= min_chars:
            base_chunks.append(current)
            current = []
    if current:
        base_chunks.append(current)

    if len(base_chunks) > 1 and _chunk_char_count(base_chunks[-1]) < min_chars:
        previous = base_chunks[-2]
        last = base_chunks[-1]
        while len(previous) > 1 and _chunk_char_count(last) < min_chars:
            candidate = previous[-1]
            if _chunk_char_count([candidate, *last]) > max_chars:
                break
            last.insert(0, previous.pop())
        if _chunk_char_count(last) < min_chars and _chunk_char_count([*previous, *last]) <= max_chars:
            base_chunks[-2] = [*previous, *last]
            base_chunks.pop()

    chunks: list[dict[str, Any]] = []
    for chunk_index, base_chunk in enumerate(base_chunks):
        chunk_units = list(base_chunk)
        if chunk_index > 0:
            overlap: list[dict[str, Any]] = []
            for unit in reversed(base_chunks[chunk_index - 1]):
                candidate = [unit, *overlap, *chunk_units]
                if _chunk_char_count(candidate) > max_chars:
                    break
                overlap.insert(0, unit)
                if _chunk_char_count(overlap) >= overlap_chars:
                    break
            chunk_units = [*overlap, *chunk_units]

        page_numbers = sorted({int(unit["page_number"]) for unit in chunk_units})
        text = " ".join(unit["text"] for unit in chunk_units).strip()
        chunks.append(
            {
                "chunk_index": chunk_index,
                "text": text,
                "page_numbers": page_numbers,
                "page_start": page_numbers[0],
                "page_end": page_numbers[-1],
            }
        )
    return chunks, threshold, distances


TEXT_UNITS = create_text_units(EXTRACTED_PAGES)
print(f"Created {len(TEXT_UNITS)} ordered paragraph/sentence units.")

# 7. Embedding generation

The same OpenAI embedding model is used for unit comparisons, final document chunks, and later query embeddings. Embeddings are generated in batches and never printed.

In [ ]:
def embed_texts(texts: Sequence[str], *, batch_size: int = 96) -> list[list[float]]:
    """Embed nonempty strings in stable batches with the configured model."""
    if not texts or any(not isinstance(text, str) or not text.strip() for text in texts):
        raise ValueError("Embedding input must contain only nonempty strings.")

    all_embeddings: list[list[float]] = []
    try:
        for start in range(0, len(texts), batch_size):
            batch = [text.replace("\n", " ") for text in texts[start : start + batch_size]]
            response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
            ordered = sorted(response.data, key=lambda item: item.index)
            all_embeddings.extend(item.embedding for item in ordered)
    except Exception as exc:
        raise RuntimeError(f"OpenAI embedding generation failed: {exc}") from exc

    if len(all_embeddings) != len(texts):
        raise RuntimeError("The embeddings API returned an unexpected result count.")
    return all_embeddings


UNIT_EMBEDDINGS = embed_texts([unit["text"] for unit in TEXT_UNITS])
SEMANTIC_CHUNKS, SEMANTIC_THRESHOLD, ADJACENT_DISTANCES = semantic_chunk_units(
    TEXT_UNITS, UNIT_EMBEDDINGS
)
CHUNK_EMBEDDINGS = embed_texts([chunk["text"] for chunk in SEMANTIC_CHUNKS])

print(f"Semantic distance threshold: {SEMANTIC_THRESHOLD:.4f}")
print(f"Created and embedded {len(SEMANTIC_CHUNKS)} final chunks.")
print("Embedding vectors are intentionally not displayed.")

# 8. ChromaDB indexing

Only `oscar_wilde_semantic_chunks` is used. Rerunning this cell refreshes that generated collection, preventing duplicates and stale records while leaving unrelated collections untouched.

In [ ]:
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
existing_names = {
    item if isinstance(item, str) else item.name
    for item in chroma_client.list_collections()
}
if COLLECTION_NAME in existing_names:
    chroma_client.delete_collection(name=COLLECTION_NAME)

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

document_fingerprint = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()[:16]
chunk_ids: list[str] = []
chunk_metadatas: list[dict[str, Any]] = []
for chunk in SEMANTIC_CHUNKS:
    text_fingerprint = hashlib.sha256(chunk["text"].encode("utf-8")).hexdigest()[:12]
    chunk_ids.append(
        f"{document_fingerprint}-chunk-{chunk['chunk_index']:04d}-{text_fingerprint}"
    )
    chunk_metadatas.append(
        {
            "source": PDF_PATH.name,
            "page_start": int(chunk["page_start"]),
            "page_end": int(chunk["page_end"]),
            "page_numbers": ",".join(str(page) for page in chunk["page_numbers"]),
            "chunk_index": int(chunk["chunk_index"]),
        }
    )

collection.add(
    ids=chunk_ids,
    documents=[chunk["text"] for chunk in SEMANTIC_CHUNKS],
    embeddings=CHUNK_EMBEDDINGS,
    metadatas=chunk_metadatas,
)
assert collection.count() == len(SEMANTIC_CHUNKS)
print(f"Indexed {collection.count()} chunks in collection: {collection.name}")

# 9. Retrieval functions

In [ ]:
NOT_FOUND_MESSAGE = "The answer is not found in the provided document."


def _validate_nonempty_string(value: str, name: str) -> str:
    """Validate and normalize a required string argument."""
    if not isinstance(value, str):
        raise TypeError(f"{name} must be a string.")
    normalized = value.strip()
    if not normalized:
        raise ValueError(f"{name} must be a nonempty string.")
    return normalized


def _retrieve_chunks(prompt: str, *, top_k: int = RETRIEVAL_TOP_K) -> list[dict[str, Any]]:
    """Retrieve the most relevant embedded chunks for a validated prompt."""
    prompt = _validate_nonempty_string(prompt, "prompt")
    if collection.count() == 0:
        return []
    query_embedding = embed_texts([prompt])[0]
    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )
    documents = (result.get("documents") or [[]])[0]
    metadatas = (result.get("metadatas") or [[]])[0]
    distances = (result.get("distances") or [[]])[0]
    retrieved: list[dict[str, Any]] = []
    for document, metadata, distance in zip(documents, metadatas, distances):
        if document:
            retrieved.append(
                {"text": document, "metadata": metadata or {}, "distance": float(distance)}
            )
    return retrieved


def _page_label(metadata: dict[str, Any]) -> str:
    """Build a compact page label from Chroma metadata."""
    page_numbers = str(metadata.get("page_numbers", "")).strip()
    if page_numbers:
        return f"Pages {page_numbers}"
    start = metadata.get("page_start")
    end = metadata.get("page_end")
    if start is None:
        return "Page unavailable"
    return f"Page {start}" if start == end else f"Pages {start}-{end}"


def _context_from_chunks(chunks: Sequence[dict[str, Any]]) -> str:
    """Format retrieved chunks as page-labelled model context."""
    return "\n\n".join(
        f"[{_page_label(chunk['metadata'])}; chunk {chunk['metadata'].get('chunk_index', '?')}]\n"
        f"{chunk['text']}"
        for chunk in chunks
    )


def retrieve_information(prompt: str) -> str:
    """Answer a nonempty prompt using only retrieved Oscar Wilde PDF context."""
    prompt = _validate_nonempty_string(prompt, "prompt")
    try:
        chunks = _retrieve_chunks(prompt)
        if not chunks:
            return NOT_FOUND_MESSAGE
        context = _context_from_chunks(chunks)
        response = openai_client.responses.create(
            model=TEXT_MODEL,
            instructions=(
                "Answer using only the supplied excerpts from the PDF The Happy Prince and Other "
                "Tales. Do not use outside knowledge. If the excerpts do not contain enough evidence, "
                f"reply exactly with: {NOT_FOUND_MESSAGE} Otherwise answer directly and include page "
                "references such as (page 4) or (pages 4, 5). Do not invent page numbers."
            ),
            input=f"Question:\n{prompt}\n\nRetrieved PDF context:\n{context}",
        )
        answer = (response.output_text or "").strip()
        return answer or NOT_FOUND_MESSAGE
    except Exception as exc:
        return f"Unable to retrieve an answer from the document: {exc}"


# 10. Structured Output parsing

The Pydantic schema has exactly two fields. `extra="forbid"` prevents additional output properties.

In [ ]:
class UserRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")

    prompt: str
    format: Literal["text", "image", "audio"]


def parse_user_request(question: str) -> UserRequest:
    """Parse a natural-language request into a clean prompt and output format."""
    question = _validate_nonempty_string(question, "question")
    response = openai_client.responses.parse(
        model=TEXT_MODEL,
        instructions=(
            "Interpret the user request for a PDF-grounded assistant. Set format to 'audio' when "
            "spoken, read-aloud, narration, voice, MP3, or audio output is requested. Set format "
            "to 'image' when an illustration, picture, drawing, or visual scene is requested. "
            "Otherwise set format to 'text'. Rewrite prompt as the underlying content request and "
            "remove all output-format instructions. Do not add facts or new requirements."
        ),
        input=question,
        text_format=UserRequest,
    )
    parsed = response.output_parsed
    if parsed is None:
        raise RuntimeError("Structured Output parsing returned no UserRequest.")
    parsed.prompt = _validate_nonempty_string(parsed.prompt, "parsed prompt")
    return parsed


assert set(UserRequest.model_fields) == {"prompt", "format"}
print("Structured Output schema fields:", sorted(UserRequest.model_fields))

# 11. Audio generation

In [ ]:
def generate_audio(answer: str) -> str:
    """Convert an answer to ElevenLabs speech, save MP3, display it, and return its path."""
    answer = _validate_nonempty_string(answer, "answer")
    api_key = load_secret("ELEVENLABS_API_KEY", required=True)
    voice_id = load_secret("ELEVENLABS_VOICE_ID", required=True)
    elevenlabs_client = ElevenLabs(api_key=api_key)
    try:
        audio_stream = elevenlabs_client.text_to_speech.convert(
            text=answer,
            voice_id=voice_id,
            model_id=ELEVENLABS_MODEL,
            output_format="mp3_44100_128",
        )
        audio_bytes = b"".join(chunk for chunk in audio_stream if chunk)
    except Exception as exc:
        raise RuntimeError(f"ElevenLabs audio generation failed: {exc}") from exc
    if not audio_bytes:
        raise RuntimeError("ElevenLabs returned an empty audio response.")

    output_path = GENERATED_DIR / f"oscar_wilde_answer_{uuid.uuid4().hex[:10]}.mp3"
    output_path.write_bytes(audio_bytes)
    display(Audio(filename=str(output_path)))
    return str(output_path)


# 12. Image generation

In [ ]:
def generate_document_illustration(prompt: str) -> str:
    """Generate and display a Victorian fairy-tale PNG grounded in retrieved PDF context."""
    prompt = _validate_nonempty_string(prompt, "prompt")
    chunks = _retrieve_chunks(prompt)
    if not chunks:
        return NOT_FOUND_MESSAGE
    context = _context_from_chunks(chunks)
    image_prompt = (
        "Create an original, document-grounded illustration for the requested scene. Base every "
        "character, action, object, and setting on the excerpts below. Use a generic Victorian "
        "fairy-tale book illustration aesthetic with delicate ink detail, subdued jewel colors, "
        "ornamental borders, expressive lighting, and no written words. Do not imitate any named "
        "or living artist. Do not introduce modern objects or unsupported story events.\n\n"
        f"Requested scene:\n{prompt}\n\nPDF excerpts:\n{context}"
    )
    try:
        response = openai_client.images.generate(
            model=IMAGE_MODEL,
            prompt=image_prompt,
            size="1024x1024",
            quality="medium",
            output_format="png",
        )
        encoded = response.data[0].b64_json if response.data else None
    except Exception as exc:
        raise RuntimeError(f"OpenAI image generation failed: {exc}") from exc
    if not encoded:
        raise RuntimeError("OpenAI returned no base64 image data.")

    output_path = GENERATED_DIR / f"oscar_wilde_illustration_{uuid.uuid4().hex[:10]}.png"
    output_path.write_bytes(base64.b64decode(encoded))
    display(Image(filename=str(output_path)))
    return str(output_path)


# 13. `ask_ai`

In [ ]:
def ask_ai(question: str) -> str:
    """Parse and route a user request, always returning text or a generated file path."""
    try:
        question = _validate_nonempty_string(question, "question")
        request = parse_user_request(question)

        if request.format == "text":
            return retrieve_information(request.prompt)
        if request.format == "audio":
            answer = retrieve_information(request.prompt)
            return generate_audio(answer)
        if request.format == "image":
            return generate_document_illustration(request.prompt)
        return "Unable to process the request: unsupported output format."
    except Exception as exc:
        return f"Unable to process the request: {exc}"


print("ask_ai is ready for text, audio, and image requests.")

# 14. Test cases

These tests intentionally make live API calls. Each test is isolated so a missing media credential, quota issue, or individual API failure does not stop the remaining cases.

In [ ]:
TEST_QUESTIONS = [
    "Answer in text: What happens to the statue of the Happy Prince?",
    "Why does the Swallow decide to remain with the Happy Prince?",
    "Which precious materials and stones decorate the Happy Prince?",
    "Explain the theme of compassion in The Happy Prince.",
    "Compare how generosity is presented in The Happy Prince and The Selfish Giant.",
    "Read aloud what the Happy Prince asks the Swallow to do for the seamstress.",
    "Create an illustration of the Swallow delivering the Prince's ruby to the seamstress.",
    "What does this book say about Oscar Wilde using a smartphone?",
]


for test_number, question in enumerate(TEST_QUESTIONS, start=1):
    print("\n" + "=" * 80)
    print(f"Test {test_number} user question: {question}")
    parsed_request: UserRequest | None = None
    try:
        parsed_request = parse_user_request(question)
        print("Parsed prompt:", parsed_request.prompt)
        print("Parsed format:", parsed_request.format)
    except Exception as parse_error:
        print("Parsed prompt: unavailable")
        print("Parsed format: unavailable")
        print("Parsing error:", parse_error)

    try:
        result = ask_ai(question)
        assert isinstance(result, str), "ask_ai must always return a string."
        print("Result:", result)

        if parsed_request and parsed_request.format in {"audio", "image"}:
            media_path = Path(result)
            assert media_path.exists(), f"Generated media path does not exist: {result}"
            if parsed_request.format == "audio":
                display(Audio(filename=str(media_path)))
            else:
                display(Image(filename=str(media_path)))
    except Exception as test_error:
        print(f"Test {test_number} failed without stopping the suite: {test_error}")

# 15. Exam checklist

- ✅ PDF is validated as smaller than 10 MB.
- ✅ PDF is included in the submission and resolved from supported Colab/repository paths.
- ✅ Semantic chunking compares adjacent OpenAI embeddings with cosine distance.
- ✅ Minimum/maximum chunk sizes and whole-unit overlap are implemented.
- ✅ ChromaDB stores explicit embeddings, unique IDs, page metadata, and chunk indices.
- ✅ Exactly one collection named `oscar_wilde_semantic_chunks` is used.
- ✅ `retrieve_information(prompt: str) -> str` exists.
- ✅ `ask_ai(question: str) -> str` exists.
- ✅ Structured Output contains exactly `prompt` and `format`.
- ✅ Text, image, and ElevenLabs audio routes are implemented.
- ✅ At least eight isolated `ask_ai()` tests are included.
- ✅ API keys are loaded from Colab Secrets or environment variables and are not hard-coded.

In [ ]:
exam_checks = {
    "PDF below 10 MB": PDF_SIZE_MB < 10,
    "PDF included/resolved": PDF_PATH.is_file(),
    "Semantic chunks created": len(SEMANTIC_CHUNKS) > 0,
    "Explicit chunk embeddings generated": len(CHUNK_EMBEDDINGS) == len(SEMANTIC_CHUNKS),
    "One required Chroma collection": collection.name == COLLECTION_NAME,
    "Chroma record count is exact": collection.count() == len(SEMANTIC_CHUNKS),
    "Structured Output has exactly two fields": set(UserRequest.model_fields) == {"prompt", "format"},
    "At least eight tests included": len(TEST_QUESTIONS) >= 8,
}

for check, passed in exam_checks.items():
    print(f"{'✅' if passed else '❌'} {check}")
assert all(exam_checks.values()), "One or more exam checklist validations failed."
print("\nAll executable exam checks passed.")